# USDA Forest Service – Pacific Southwest Research Station, Institute of Pacific Islands Forestry (IPIF) & The Nature Conservancy (TNC) Addax Data Science V1 (HWI-ADSV1) Sagemaker Serverless Deployment¶

This notebook deploys the Hawaii Addax Data Science V1 classifier to a Sagemaker serverless endpoint. It is intended to be run in a SageMaker Notebook instance on the conda_pytorch_p10 kernel.

## Setup

In [2]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import time
import json
import base64
from datetime import datetime

## Initialize AWS Session

In [3]:
sess = boto3.Session()
sm = sess.client('sagemaker')
region = sess.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

## Get IAM Role

Note: Ensure the IAM role has:

- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`

In [4]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

Using role: arn:aws:iam::830244800171:role/service-role/AmazonSageMaker-ExecutionRole-20210125T212674


## Create ECR Repository

In [5]:
# Create ECR repository if it doesn't exist
registry_name = "hwi-adsv1-sagemaker-serverless"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ECR repository already exists")
    pass

## Build and Upload Container

Builds the the inference container with hwi-adsv1 and the sagemaker handler and uploads it to ECR. This step takes several minutes after it prints the 'Login Succeeded' message. Be patient and trust the process.

In [6]:
# flag to avoid timely image builds
should_create = True

if should_create:
    # Get auth token and login to ECR
    !aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com
    
    # Build container
    !docker build -q -t {registry_name} -f Dockerfile .
    
    # Tag and push to ECR
    image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"
    !docker tag {registry_name} {image_uri}
    !docker push {image_uri}
    
    print(f"Container pushed to: {image_uri}")

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
sha256:c463b9a06c25efea22a20accd33647e22433f450dbd08db8be823add75303f76
The push refers to repository [830244800171.dkr.ecr.us-west-2.amazonaws.com/hwi-adsv1-sagemaker-serverless]

bf18a086: Preparing 
3909e6c3: Preparing 
1fc2e882: Preparing 
aea02905: Preparing 
b5fc88b3: Preparing 
640cda86: Preparing 
207b52a4: Preparing 
143f3d7f: Preparing 
2b569c3a: Preparing 
latest: digest: sha256:db1fba950b40cd42311dfd7706ab149b1e013306a24bbbb8fe6e2836d0d0783b size: 2419
Container pushed to: 830244800171.dkr.ecr.us-west-2.amazonaws.com/hwi-adsv1-sagemaker-serverless:latest


## Create Sagemaker Model

In [7]:
model_prefix = "hwi-adsv1"

# Check if model already exists
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    create_model_response = sm.create_model(
        ModelName=model_prefix,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": image_uri,
            "Environment": {
                "SAGEMAKER_PROGRAM": "serve.py"
            }
        }
    )

print(f"Model ARN: {create_model_response['ModelArn']}")

Model ARN: arn:aws:sagemaker:us-west-2:830244800171:model/hwi-adsv1


## Create Sagemaker Realtime Endpoint

In [8]:
# Create realtime and batch endpoint configuration
realtime_endpoint_config_name = f"{model_prefix}-realtime-config"

realtime_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=realtime_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 6144,  # 6GB memory
                "MaxConcurrency": 20       # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Realtime endpoint config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")

Realtime endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/hwi-adsv1-realtime-config


## Create Realtime Endpoint

In [9]:
# Create realtime endpoint
realtime_endpoint_name = f"{model_prefix}-concurrency-20"
create_realtime_endpoint_response = sm.create_endpoint(
    EndpointName=realtime_endpoint_name,
    EndpointConfigName=realtime_endpoint_config_name
)

print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")

# Wait for endpoint creation
resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
realtime_status = resp['EndpointStatus']
print(f"Status: {realtime_status}")

while realtime_status == 'Creating':
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
        print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
        break

# Get CloudWatch logs for the endpoint
logs = boto3.client('logs')

print(f"Realtime Arn: {resp['EndpointArn']}")
print(f"Realtime endpoint final status: {realtime_status}")
if realtime_status == 'Failed':
    realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
    try:
        log_streams = logs.describe_log_streams(
            logGroupName=realtime_log_group,
            orderBy='LastEventTime',
            descending=True,
            limit=1
        )
        if log_streams['logStreams']:
            stream = log_streams['logStreams'][0]
            print(f"\nLog stream: {stream['logStreamName']}")
            realtime_events = logs.get_log_events(
                logGroupName=realtime_log_group,
                logStreamName=stream['logStreamName'],
                startFromHead=True
            )
            for event in realtime_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

Endpoint ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint/hwi-adsv1-concurrency-20
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Realtime Arn: arn:aws:sagemaker:us-west-2:830244800171:endpoint/hwi-adsv1-concurrency-20
Realtime endpoint final status: InService


## Create Batch Endpoint Config

In [10]:
batch_endpoint_config_name = f"{model_prefix}-batch-config"

# Disable batch endpoint config creation if not needed
create_batch_endpoint_config = True

if create_batch_endpoint_config:
    batch_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=batch_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_prefix,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 6144,  # 6GB memory
                    "MaxConcurrency": 80       # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Batch endpoint config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")

Batch endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/hwi-adsv1-batch-config


## Create Batch Endpoint

In [11]:
# Create batch endpoint
create_batch_endpoint = True

if create_batch_endpoint:
    batch_endpoint_name = f"{model_prefix}-concurrency-80"
    create_batch_endpoint_response = sm.create_endpoint(
        EndpointName=batch_endpoint_name,
        EndpointConfigName=batch_endpoint_config_name
    )
    
    print(f"Endpoint ARN: {create_batch_endpoint_response['EndpointArn']}")
    
    # Wait for endpoint creation
    resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
    batch_status = resp['EndpointStatus']
    print(f"Status: {batch_status}")
    
    while batch_status == 'Creating':
        time.sleep(60)
        resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
        batch_status = resp['EndpointStatus']
        print(f"Status: {batch_status}")
        if batch_status == 'Failed':
            batch_failure_reason = resp.get('FailureReason', 'No failure reason provided')
            print(f"Batch endpoint deployment failed: {batch_failure_reason}")
            break
    
    # Get CloudWatch logs for the endpoint
    logs = boto3.client('logs')
    
    print(f"Batch Arn: {resp['EndpointArn']}")
    print(f"Batch endpoint final status: {batch_status}")
    if batch_status == 'Failed':
        batch_log_group = f"/aws/sagemaker/Endpoints/{batch_endpoint_name}"
        try:
            log_streams = logs.describe_log_streams(
                logGroupName=batch_log_group,
                orderBy='LastEventTime',
                descending=True,
                limit=1
            )
            if log_streams['logStreams']:
                stream = log_streams['logStreams'][0]
                print(f"\nLog stream: {stream['logStreamName']}")
                batch_events = logs.get_log_events(
                    logGroupName=batch_log_group,
                    logStreamName=stream['logStreamName'],
                    startFromHead=True
                )
                for event in batch_events['events']:
                    print(event['message'])
        except Exception as e:
            print(f"Error fetching logs: {str(e)}")

Endpoint ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint/hwi-adsv1-concurrency-80
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Batch Arn: arn:aws:sagemaker:us-west-2:830244800171:endpoint/hwi-adsv1-concurrency-80
Batch endpoint final status: InService
